# Chạy thử: LLaVA-OneVision-7B, BLIP-2, Qwen3-VL-8B

Mục đích: xác nhận ba mô hình chạy được trên Colab, đo bộ nhớ GPU và thời gian thực tế, xem vài caption mẫu.

**Trước khi chạy:** Runtime → Change runtime type → chọn **T4 GPU** → Save. Sau đó chạy lần lượt từng ô từ trên xuống.

Cả ba mô hình đều nạp **4-bit** để so sánh công bằng. Kết quả được lưu lại sau mỗi mô hình.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cài thư viện

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

## 3. Chuẩn bị ảnh và hàm chạy

In [ ]:
import gc, io, json, time, urllib.request

import matplotlib.pyplot as plt
import torch
import transformers
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

assert torch.cuda.is_available(), "Chưa có GPU: chọn Runtime > Change runtime type > T4 GPU"
print("transformers", transformers.__version__, "| GPU", torch.cuda.get_device_name(0))

PROMPT = "Describe this image in one short sentence."
MAX_NEW_TOKENS = 60
RESULT_FILE = "/content/ketqua_chay_thu.json"
results = {}


def load_url(image_id):
    url = f"http://images.cocodataset.org/val2017/{image_id}.jpg"
    with urllib.request.urlopen(url, timeout=30) as response:
        return Image.open(io.BytesIO(response.read())).convert("RGB")


def to_image(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def bar_chart():
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(["Model A", "Model B", "Model C"], [72, 85, 64], color="#4C78A8")
    ax.set_title("Caption accuracy of three models")
    ax.set_xlabel("Model")
    ax.set_ylabel("Accuracy (%)")
    return to_image(fig)


def line_chart():
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot([2021, 2022, 2023, 2024, 2025], [10, 18, 31, 47, 70], marker="o", color="#4C78A8")
    ax.set_title("Number of papers per year")
    ax.set_xlabel("Year")
    ax.set_ylabel("Papers")
    ax.set_xticks([2021, 2022, 2023, 2024, 2025])
    return to_image(fig)


# Biểu đồ tự vẽ nên biết chắc nội dung đúng, dễ phát hiện mô hình mô tả sai.
images = {
    "coco_000000039769": load_url("000000039769"),
    "coco_000000000285": load_url("000000000285"),
    "coco_000000000632": load_url("000000000632"),
    "coco_000000000724": load_url("000000000724"),
    "coco_000000000776": load_url("000000000776"),
    "bar_chart (A=72, B=85, C=64)": bar_chart(),
    "line_chart (10, 18, 31, 47, 70; tăng dần)": line_chart(),
}

for name, image in images.items():
    print(name, image.size)


def caption_chat(model, processor, image):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT},
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device, torch.float16)
    output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


def caption_blip2(model, processor, image):
    # BLIP-2 sinh caption không cần prompt: đây là cách dùng chuẩn của mô hình.
    inputs = processor(images=image, return_tensors="pt").to(model.device, torch.float16)
    output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return processor.batch_decode(output, skip_special_tokens=True)[0].strip()


def gb(num_bytes):
    return round(num_bytes / 1024 ** 3, 2)


def run_model(name, model_id, caption_fn):
    record = {"model_id": model_id, "quantization": "4-bit nf4", "prompt": PROMPT, "captions": {}}
    results[name] = record
    model = processor = None
    try:
        torch.cuda.reset_peak_memory_stats()
        start = time.time()
        processor = AutoProcessor.from_pretrained(model_id)
        model = AutoModelForImageTextToText.from_pretrained(
            model_id,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            ),
            device_map="auto",
        )
        record["load_seconds"] = round(time.time() - start, 1)
        record["gpu_gb_after_load"] = gb(torch.cuda.max_memory_allocated())
        print(f"Đã nạp {name}: {record['gpu_gb_after_load']} GB, {record['load_seconds']} giây")

        for image_name, image in images.items():
            start = time.time()
            text = caption_fn(model, processor, image)
            seconds = round(time.time() - start, 1)
            record["captions"][image_name] = {"text": text, "seconds": seconds}
            print(f"\n[{image_name}] ({seconds} giây)\n{text}")

        record["gpu_gb_peak"] = gb(torch.cuda.max_memory_allocated())
        record["status"] = "ok"
    except Exception as error:
        record["status"] = f"LỖI {type(error).__name__}: {error}"
        print(record["status"])
    finally:
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()
        with open(RESULT_FILE, "w", encoding="utf8") as file:
            json.dump(results, file, ensure_ascii=False, indent=2)
    print(f"\n{name}: {record['status']} | GPU cao nhất: {record.get('gpu_gb_peak')} GB")

## 4. LLaVA-OneVision-7B (mô hình chính)

In [ ]:
run_model("LLaVA-OneVision-7B", "llava-hf/llava-onevision-qwen2-7b-ov-hf", caption_chat)

## 5. BLIP-2 OPT 2.7B (đối chứng)

In [ ]:
run_model("BLIP-2 OPT 2.7B", "Salesforce/blip2-opt-2.7b", caption_blip2)

## 6. Qwen3-VL-8B (đối chứng thế hệ mới)

In [ ]:
run_model("Qwen3-VL-8B", "Qwen/Qwen3-VL-8B-Instruct", caption_chat)

## 7. Tổng kết và tải kết quả

In [ ]:
for name, record in results.items():
    times = [item["seconds"] for item in record["captions"].values()]
    average = round(sum(times) / len(times), 1) if times else None
    print(f"{name}: {record['status'][:60]} | nạp {record.get('gpu_gb_after_load')} GB | "
          f"cao nhất {record.get('gpu_gb_peak')} GB | trung bình {average} giây/ảnh")

from google.colab import files
files.download(RESULT_FILE)